<div style="background-color: #FFDDDD; border-left: 5px solid red; padding: 10px; color: black;">
    <strong>Kernel:</strong> Python 3 (ipykernel)
</div>

# 🚀 Deploy Stable Diffusion 3.5 Medium with LoRA adapters on Amazon SageMaker

## Prerequisites

To start off, let's install some packages to help us through the notebooks. **Restart the kernel after packages have been installed.**

In [ ]:
%pip install sagemaker=="3.4.1"
%pip install -r inference/code/requirements.txt --upgrade --quiet

## This cell will restart the kernel. Click "OK".

In [ ]:
from IPython import get_ipython
get_ipython().kernel.do_shutdown(True)

***

Initialize some of the basic variables that you will use during this section.

In [ ]:
import os
import json
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.config import load_sagemaker_config
from sagemaker.core.common_utils import name_from_base

sagemaker_session = Session()
s3_client = boto3.client('s3')

region = sagemaker_session.boto_session.region_name
bucket = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix
configs = load_sagemaker_config()

role = get_execution_role(sagemaker_session, use_default=True)

print(f"Execution Role: {role}")
print(f"Default S3 Bucket: {bucket}")

## Deploy Model to SageMaker Hosting

### Step 1: Get HuggingFace Pytorch Container to host Stable Diffusion

Since you've already explored how to use a custom container for training, you'll learn how to use a managed container for inference here. A full reference of all the available managed SageMaker inference containers can be found in the [AWS DeepLearning Containers available images repository](https://aws.github.io/deep-learning-containers/reference/available_images/).

In [ ]:
inference_image_uri=f"763104351884.dkr.ecr.{region}.amazonaws.com/huggingface-pytorch-inference:2.6.0-transformers4.51.3-gpu-py312-cu124-ubuntu22.04"

print(f"Using image to host: {inference_image_uri}")

### Step 2: Deploy model

If you've already run training, then this step is a repeat of downloading the model. It will find cached versions and complete almost immediately. If you haven't, then the model will get downloaded from HuggingFace so you can use it for inference.

**If you entered your HuggingFace token during training, it will be retrieved from the stored session. You do not need to enter it again.**

In [ ]:
import os

HF_TOKEN = ""

%store -r HF_TOKEN

assert HF_TOKEN.startswith("hf_"), "A valid HuggingFace API Key is required"

os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
model_id = "stabilityai/stable-diffusion-3.5-medium"
model_id_filesafe = model_id.replace("/","_")

use_local_model = True

Download the model from HuggingFace and store in S3 (will not re-download if already cached/uploaded)

In [ ]:
%%time

from huggingface_hub import snapshot_download
import os
import subprocess

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

if use_local_model:
    model_local_location = f"../models/{model_id_filesafe}"
    prefix = f"{default_prefix}/" if default_prefix else ""
    model_s3_destination = f"s3://{bucket}/{prefix}models/{model_id_filesafe}/"
    
    print(f"Downloading model {model_id}")
    os.makedirs(model_local_location, exist_ok=True)
    #snapshot_download(repo_id=model_id, local_dir=model_local_location)

    subprocess.run(['hf', 'download', '--local-dir', model_local_location, model_id])
    
    print("Beginning Model Upload...")
    subprocess.run(['aws', 's3', 'sync', model_local_location, model_s3_destination, 
                   '--exclude', '.cache/*', '--exclude', '.gitattributes'])
    
    print(f"Model uploaded to:\n{model_s3_destination}")
    os.environ["model_location"] = model_s3_destination
else:
    os.environ["model_location"] = model_id

## Upload custom code and LoRA adapters

The `lora_adapters` directory contains the LoRA adapters that will be used for inference. The example repository comes with a sample that was trained more epochs than you have in this workshop to be used as reference.

The `inference_code` directory contains the custom inference script along with a requirements.txt file containing custom package versions. To view the custom inference script, open `inference/code/inference.py`.

You will copy both of these folders to the S3 folder containing the base model, then load it for inference. The hosting container will automatically load the custom code to override defaults functions, which will handle adapter loading as well.

⚠️ **Note:** Take a moment to look at both the structure of the adapters folder and the custom inference code to better understand how it all works.

In [ ]:
!aws s3 cp --recursive inference $model_s3_destination --exclude **.ipynb_checkpoints/**
!aws s3 cp --recursive lora_adapters $model_s3_destination --exclude **.ipynb_checkpoints/**

Create a model object that will be used for deployment to your SageMaker Endpoint. The model object contains the reference to the ECR container that should be used to host the model, as well as the S3 path to the model artifact. Since you have uploaded your LoRA adapters and custom inference scripts in a previous step, the container will automatically load these at runtime.

In [ ]:
from sagemaker.core.resources import Model, Endpoint, EndpointConfig
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant, ModelDataSource, S3ModelDataSource

model_name = "stable-diffusion-35-medium"

#Add any custom inference environment configuration

inference_llm_config = {
}

core_model = Model.create(
    model_name=model_name,

    execution_role_arn=role,
    primary_container=ContainerDefinition(
        image=inference_image_uri,
        environment=inference_llm_config,
        model_data_source=ModelDataSource(
            s3_data_source=S3ModelDataSource(
                s3_uri=os.environ["model_location"],
                s3_data_type="S3Prefix",
                compression_type="None",
            )
        )
    ),
)

Now deploy the model to an endpoint. Here you can configure the number of instances used for hosting as well as the desired instance types. Additionally you could configure autoscaling policies.

As this runs, you will see dashes printed every 30 seconds, indicating that deployment is in progress. 

⚠️⚠️**Do not re-run the cell.**⚠️⚠️

**The total time for the endpoint to respond as `InService` is around 10 minutes. This includes spinning up infrastructure, loading the serving container, loading the model, and loading any adapters.**

In [ ]:
from sagemaker.core.helper.session_helper import _wait_until, _deploy_done

endpoint_name = f"{model_name}-endpoint"
BASE_ENDPOINT_NAME = name_from_base(endpoint_name)

EndpointConfig.create(
    endpoint_config_name=BASE_ENDPOINT_NAME,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=model_name,
            initial_instance_count=1,
            instance_type="ml.g5.2xlarge",
        )
    ],
)

core_endpoint = Endpoint.create(
    endpoint_name=BASE_ENDPOINT_NAME,
    endpoint_config_name=BASE_ENDPOINT_NAME,
)

_wait_until(lambda: _deploy_done(sagemaker_session.sagemaker_client, BASE_ENDPOINT_NAME), poll=30)
core_endpoint = Endpoint.get(endpoint_name=BASE_ENDPOINT_NAME)
print(f"\n\nEndpoint status: {core_endpoint.endpoint_status}")

## Visualizing the output

Now that the model is deployed, you will run through some examples of invoking the model. Since you deployed the base model without fusing the LoRA adapters back in, you can host 1 copy of the model and invoke any of the adapters at runtime, including the base model itself. To use the base model, simply omit the `adapter` configuration in your input parameter payload.


These are helper functions to decode and render the output images

In [ ]:
from PIL import Image
from io import BytesIO
from IPython.display import display
import base64
import matplotlib.pyplot as plt

def decode_base64_image(image_string):
  base64_image = base64.b64decode(image_string)
  buffer = BytesIO(base64_image)
  return Image.open(buffer)
 
# display PIL images as grid
def display_images(images=None,columns=3, width=30, height=30):
    plt.figure(figsize=(width, height))
    for i, image in enumerate(images):
        plt.subplot(int(len(images) / columns + 1), columns, i + 1)
        plt.axis('off')
        plt.imshow(image)

Now supply an input prompt and inference parameters for generation.

Inside of the `parameters` array, you can add/remove the `adapter` field to toggle between the base model and fine tuned adapters. The names of the adapters will correlate to the names of the folders inside of `lora_adapters/adapters` on your filesystem. If you changed the names of any of them prior to deployment, you will need to make the corresponding changes to match below or you will get errors.

The first example will generate from the base, then the adapter you tuned in this workshop, and finally an adapter that was trained for longer on a bigger training instance. Stable Diffusion benefits greatly from longer training runs (thousands of epochs) to refine output quality. The longer running example is just a small sample and would be even more improved from a longer training time.

In [ ]:
prompt = "A boy riding his bike in the neighborhood with his dog."

## Ground Truth
This shows a sample image from the ground truth dataset

In [ ]:
from datasets import load_from_disk
dataset = load_from_disk("data")
gt_image = dataset['train'][1]['image']
display_images([gt_image])

## Base Model

When viewing the base model output, you notice that the model looks nothing like the source material from training.

In [ ]:
response = core_endpoint.invoke(
    body=json.dumps({
        "inputs": prompt,
        "parameters": {
            "num_inference_steps": 30,
            "guidance_scale": 7.0,
            "num_images_per_prompt": 1
        }
    }),
    content_type="application/json",
    accept="application/json",
)

result = json.loads(response.body.read().decode("utf-8"))

base_decoded_images = [decode_base64_image(image) for image in result["generated_images"]]

display_images(base_decoded_images)

## Tuned Adapter from this workshop (~15 mins on `ml.g5.12xlarge`)

Here, while not quite like the source material, the output is starting to resemble the style of the ground truth output.

In [ ]:
response = core_endpoint.invoke(
    body=json.dumps({
        "inputs": prompt,
        "parameters": {
            "adapter":"workshop-adapter",
            "num_inference_steps": 30,
            "guidance_scale": 7.0,
            "num_images_per_prompt": 1
        }
    }),
    content_type="application/json",
    accept="application/json",
)

result = json.loads(response.body.read().decode("utf-8"))

workshop_decoded_images = [decode_base64_image(image) for image in result["generated_images"]]

display_images(workshop_decoded_images)

## Pre-tuned adapter on a larger instance (~1hr on `ml.p4d.24xlarge`)

This example on an adapter that was trained on a larger instance for longer, starts to look much more like ground truth but not exactly perfect either.

In [ ]:
response = core_endpoint.invoke(
    body=json.dumps({
        "inputs": prompt,
        "parameters": {
            "adapter":"boy-cartoon",
            "num_inference_steps": 30,
            "guidance_scale": 7.0,
            "num_images_per_prompt": 1
        }
    }),
    content_type="application/json",
    accept="application/json",
)

result = json.loads(response.body.read().decode("utf-8"))

tuned_decoded_images = [decode_base64_image(image) for image in result["generated_images"]]

display_images(tuned_decoded_images)

Finally, a side-by-side comparison of all 4 samples.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data and titles
images = [gt_image, base_decoded_images[0], workshop_decoded_images[0], tuned_decoded_images[0]]
titles = ['Ground Truth', 'Base', 'Workshop', 'Tuned']

# Create subplots
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Plot images
for ax, img, title in zip(axes, images, titles):
    ax.imshow(np.array(img))
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Congratulations, you've successfully fine-tuned and hosted your Stable Diffusion 3.5 medium model!